# 04 · RAG over Gutenberg (Sprint 5)

Локальный LLM (Ollama + qwen3:14b) поверх multilingual ChromaDB index 1828 книг Project Gutenberg. См. [SESSION_2026-05-15-rag.md](../SESSION_2026-05-15-rag.md), [rag.md](../rag.md), [decisions.md](../decisions.md) D14/D15.

Каждый ответ возвращается с цитатами `[Автор, "Название", PG12345]` + структурированный список sources снизу. Если LLM сгенерил PG-id которого нет в retrieval, появляется warning.

In [ ]:
import sys
sys.path.insert(0, '/workspace/scripts')
from rag_query import rag_query
from IPython.display import Markdown, display

def ask(question, k=8, model='qwen3:14b'):
    r = rag_query(question, k=k, model=model)
    md = f"### {question}\n\n{r['answer']}\n\n"
    md += f"<details><summary><b>Sources ({len(r['sources'])})</b> · retrieval {r['timing']['retrieval_s']}s · generation {r['timing']['generation_s']}s · tokens {r['tokens']}</summary>\n\n"
    for s in r['sources']:
        md += f"- **[{s['pg_id']}]** {s['author']} — *{s['title']}* (chunk {s['chunk']}, dist {s['distance']})  \n"
        md += f"  > {s['snippet'][:240].strip()}...\n"
    md += "\n</details>"
    if r['warnings']:
        md += "\n\n⚠ " + "; ".join(r['warnings'])
    display(Markdown(md))
    return r

## 1 · «найди упоминания битой посуды в книгах» (русский, образный)

In [ ]:
_ = ask("найди упоминания битой посуды в книгах")

## 2 · «где Дживс выручает Берти из неловкой ситуации?» (русский + имена)

In [ ]:
_ = ask("где Дживс выручает Берти из неловкой ситуации?")

## 3 · «какие фирменные словечки у Wodehouse означают опьянение?» (стилометрия + язык)

In [ ]:
_ = ask("какие фирменные словечки у Wodehouse означают опьянение?")

## 4 · `what are typical Wodehouse exclamations like 'what ho'?` (English RAG)

In [ ]:
_ = ask("what are typical Wodehouse exclamations like 'what ho'?")

## 5 · `describe how Sherlock Holmes uses tobacco` (English, другой автор)

In [ ]:
_ = ask("describe how Sherlock Holmes uses tobacco")

## 6 · `статистика блокчейн транзакций по годам` (edge case — НЕТ ответа в корпусе)

Smoke-test honesty: модель должна признать что в корпусе нет ничего про блокчейн, а не сочинить.

In [ ]:
_ = ask("статистика блокчейн транзакций по годам")

## ⚙ Произвольный запрос

Поменяй вопрос и Shift+Enter:

In [ ]:
_ = ask("опиши типичную сцену в кабинете лорда", k=8)